# Практическая работа №3
**Тема:** Очистка и исследование данных о продажах видеоигр
**Выполнил:** Шумилин Демьян

## Задание 1. Загрузка данных

In [1]:
# Импортируем библиотеки для работы с таблицами и построения графиков
import pandas as pd
import matplotlib.pyplot as plt

# Считываем CSV-файл с датасетом
df = pd.read_csv('vgsales.csv')

# Выводим первые 5 строк, чтобы оценить структуру таблицы
print(df.head(5))

   Rank                        Name Platform    Year         Genre Publisher  NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales
0     1                  Wii Sports     Wii  2006.00         Sports  Nintendo     41.49     29.02      3.77         8.46         82.74
1     2           Super Mario Bros.    NES  1985.00        Platform  Nintendo     29.08      3.58      6.81         0.77         40.24
2     3              Mario Kart Wii     Wii  2008.00         Racing  Nintendo     15.85     12.88      3.79         3.31         35.82
3     4           Wii Sports Resort     Wii  2009.00         Sports  Nintendo     15.75     11.01      3.28         2.96         33.00
4     5    Pokemon Red/Pokemon Blue     GB  1996.00 Role-Playing  Nintendo     11.27      8.89     10.22         1.00         31.37


Подключили библиотеки pandas и matplotlib, загрузили данные и вывели первые 5 строк.

## Задание 2. Исследование структуры данных

In [2]:
# Смотрим общее число строк и колонок
print("Размер датасета:", df.shape)

# Выводим список всех наименований колонок
print("\nНазвания столбцов:", list(df.columns))

# Проверяем типы данных в каждом столбце
print("\nТипы данных:")
print(df.dtypes)

# Рассчитываем ключевую статистику (среднее, мин, макс) по региональным продажам
print("\nСтатистика по продажам (млн копий):")
print(df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']].describe())

Размер датасета: (16598, 11)

Названия столбцов: ['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

Типы данных:
Rank              int64
Name             object
Platform         object
Year            float64
Genre            object
Publisher        object
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object

Статистика по продажам (млн копий):
           NA_Sales      EU_Sales      JP_Sales   Other_Sales  Global_Sales
count  16598.000000  16598.000000  16598.000000  16598.000000  16598.000000
mean       0.264667      0.146652      0.077782      0.048063      0.537441
std        0.816683      0.505351      0.309291      0.188588      1.555028
min        0.000000      0.000000      0.000000      0.000000      0.010000
max       41.490000     29.020000     10.220000     10.570000     82.740000


**Выводы по Заданию 2:**
* **Числовые столбцы:** `Rank`, `Year`, `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`, `Global_Sales`
* **Текстовые (категориальные) столбцы:** `Name`, `Platform`, `Genre`, `Publisher`
* **Столбцы с продажами:** `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`, `Global_Sales`

## Задание 3. Поиск пропущенных значений

In [3]:
# Считаем сумму пустых значений (NaN) для каждого столбца
null_counts = df.isnull().sum()
print("Пропуски по столбцам:\n", null_counts)

# Автоматически определяем столбец с наибольшим числом пропусков через idxmax()
print(f"\nБольше всего пропусков в столбце: {null_counts.idxmax()} ({null_counts.max()})")

Пропуски по столбцам:
Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

Больше всего пропусков в столбце: Year (271)


**Ответ к Заданию 3:** Пропуски присутствуют в столбцах `Year` (271) и `Publisher` (58). Наибольшее количество пропусков — в столбце `Year`.

## Задание 4. Очистка данных

In [4]:
# Удаляем записи, в которых отсутствует год релиза, и сохраняем в df_clean
df_clean = df.dropna(subset=['Year']).copy()

# Сравниваем размер датасета до и после очистки
print(f"Количество строк в исходном df: {len(df)}")
print(f"Количество строк в очищенном df_clean: {len(df_clean)}")
print(f"Удалено строк с пропущенным годом: {len(df) - len(df_clean)}")

Количество строк в исходном df: 16598
Количество строк в очищенном df_clean: 16327
Удалено строк с пропущенным годом: 271


**Ответы к Заданию 4:**
1. В исходном датасете было **16 598** строк.
2. В очищенном датасете осталось **16 327** строк.
3. Пропущенные значения мешают правильному анализу временных рядов, так как отсутствие года не позволяет корректно сгруппировать продажи по годам.

## Задание 5. Изучение категориальных признаков

In [5]:
# Считаем число уникальных категорий через .nunique()
print("Количество уникальных жанров (Genre):", df_clean['Genre'].nunique())
print("Количество уникальных платформ (Platform):", df_clean['Platform'].nunique())
print("Количество уникальных издателей (Publisher):", df_clean['Publisher'].nunique())

Количество уникальных жанров (Genre): 12
Количество уникальных платформ (Platform): 31
Количество уникальных издателей (Publisher): 576


**Ответ к Заданию 5:** В датасете 12 жанров, 31 платформа и 576 издателей. Больше всего уникальных значений у признака **Publisher**.

## Задание 6 и 7. Анализ и продажи по жанрам

In [6]:
# Считаем количество выходивших игр для каждого жанра
print("--- Количество игр по жанрам ---")
print(df_clean['Genre'].value_counts())

# Группируем по жанру и суммируем мировые продажи, затем сортируем по убыванию
print("\n--- Мировые продажи по жанрам (млн) ---")
print(df_clean.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=False))

--- Количество игр по жанрам ---
Action          3251
Sports          2304
Misc            1686
Role-Playing    1470
Shooter         1282
Adventure       1274
Racing          1225
Platform         875
Simulation       848
Fighting         836
Strategy         670
Puzzle           570
Name: Genre, dtype: int64

--- Мировые продажи по жанрам (млн) ---
Genre
Action          1722.84
Sports          1309.24
Shooter         1026.18
Role-Playing     923.84
Platform         827.99
Misc             789.87
Racing           726.76
Fighting         444.05
Simulation       389.88
Puzzle           242.22
Adventure        234.59
Strategy         173.27
Name: Global_Sales, dtype: float64


**Ответ к Заданиям 6-7:** Абсолютным лидером как по количеству выпущенных игр (3251), так и по суммарным мировым продажам (1722.84 млн) является жанр **Action**.

## Задание 8 и 9. Самые продаваемые игры и анализ платформ

In [7]:
# Находим топ-5 самых успешных игр через сортировку по Global_Sales
print("--- Топ-5 самых продаваемых игр ---")
print(df_clean.sort_values(by='Global_Sales', ascending=False)[['Name', 'Platform', 'Genre', 'Global_Sales']].head())

# Суммируем продажи для каждой игровой платформы и выводим 5 лучших
print("\n--- Топ-5 платформ по суммарным продажам (млн) ---")
print(df_clean.groupby('Platform')['Global_Sales'].sum().sort_values(ascending=False).head())

--- Топ-5 самых продаваемых игр ---
                       Name Platform    Genre  Global_Sales
0                Wii Sports     Wii   Sports         82.74
1         Super Mario Bros.    NES  Platform         40.24
2            Mario Kart Wii     Wii   Racing         35.82
3         Wii Sports Resort     Wii   Sports         33.00
4  Pokemon Red/Pokemon Blue     GB Role-Playing         31.37

--- Топ-5 платформ по суммарным продажам (млн) ---
Platform
PS2     1233.46
X360     969.60
PS3      949.74
Wii      909.81
DS       818.91
Name: Global_Sales, dtype: float64


**Ответ к Заданиям 8-9:** Самая продаваемая игра в истории — **Wii Sports** (82.74 млн). Самая успешная платформа по совокупным продажам — **PS2** (1233.46 млн).

## Задание 10. Визуализация продаж по жанрам

In [8]:
# Агрегируем данные продаж по жанрам и сортируем по возрастанию для эстетичного графика
genre_sales = df_clean.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=True)

# Настраиваем размер и строи горизонтальную столбчатую диаграмму
plt.figure(figsize=(9, 4.5))
genre_sales.plot(kind='barh', color='#2b5c8f')

# Добавляем заголовки и подписи осей
plt.title('Мировые продажи по жанрам (млн копий)')
plt.xlabel('Продажи, млн')
plt.ylabel('Жанр')
plt.tight_layout()
plt.show()

**Ответы к Заданию 10:**
1. **Наибольшие продажи:** `Action` (1722.84 млн) и `Sports` (1309.24 млн).
2. **Наименьшие продажи:** `Strategy` (173.27 млн) и `Adventure` (234.59 млн).
3. Разрыв между жанром-лидером (Action) и замыкающим (Strategy) составляет почти 10 раз.

## Задание 11. Корреляция продаж

In [9]:
# Рассчитываем коэффициент линейной корреляции Пирсона между продажами в США и в мире
corr_na_global = df_clean['NA_Sales'].corr(df_clean['Global_Sales'])
print(f"Коэффициент корреляции между NA_Sales и Global_Sales: {corr_na_global:.4f}")

Коэффициент корреляции между NA_Sales и Global_Sales: 0.9413


**Ответы к Заданию 11:**
* Коэффициент корреляции **положительный** (`0.9413`).
* Значение чрезвычайно близко к 1, что говорит о наисильнейшей прямой связи между продажами в Северной Америке и мировыми продажами.

## Задание 12. Диаграмма рассеяния

In [10]:
# Строим диаграмму рассеяния, чтобы визуально оценить линейную взаимосвязь
plt.figure(figsize=(8, 5))
plt.scatter(df_clean['NA_Sales'], df_clean['Global_Sales'], alpha=0.5, color='#2b5c8f')

# Оформляем заголовок и оси графика
plt.title('Диаграмма рассеяния: NA_Sales vs Global_Sales')
plt.xlabel('Продажи в Северной Америке, млн')
plt.ylabel('Мировые продажи, млн')
plt.tight_layout()
plt.show()

**Ответы к Заданию 12:**
1. Каждая точка — это отдельная видеоигра.
2. Направление облака точек снизу-слева вверх-направо подтверждает выраженную положительную связь.
3. График полностью наглядно подтверждает полученный выше высокий коэффициент корреляции (0.9413).

## Задание 13. Корреляция между регионами

In [11]:
# Задаем список колонок с региональными продажами
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

# Строим матрицу парной корреляции между всеми регионами
print(df_clean[sales_cols].corr().round(2))

              NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales
NA_Sales          1.00      0.77      0.45         0.63          0.94
EU_Sales          0.77      1.00      0.44         0.73          0.90
JP_Sales          0.45      0.44      1.00         0.29          0.61
Other_Sales       0.63      0.73      0.29         1.00          0.75
Global_Sales      0.94      0.90      0.61         0.75          1.00


**Ответ к Заданию 13:** Самая сильная корреляция наблюдается между `NA_Sales` и `EU_Sales` (**0.77**). Самая слабая связь с остальными регионами наблюдается у рынка Японии (`JP_Sales`), что вызвано уникальной культурной спецификой японского игрового рынка.